# Predict Meal

## Import Libraries

In [3]:
import pickle
import pandas as pd
import numpy as np
import os

## Load Model

In [4]:
# ==============================================================================
# 1. MODEL INITIALIZATION & RESOURCE LOADING
# ==============================================================================
# Objective: Load the pre-trained K-Nearest Neighbors (KNN) model and database.
#
# - 'model_meal.pickle' contains:
#   1. knn_model: The trained algorithm to find similar foods.
#   2. scaler: The scaler used to normalize input data (crucial for accurate predictions).
#   3. meal_db: The reference database containing food items and nutrition info.
#   4. features: The exact list of columns used during training.

# Define path (Update this to your actual path if needed)
file_path = '../../models/model_meal.pickle' 
# Or use the absolute path you provided:
# file_path = r'C:\Users\vince\Documents\PPTI\CAWU_4_VincentF\VIKTORIA\project\viktorifit-ml\models\model_meal.pickle'

try:
    with open(file_path, 'rb') as f:
        meal_data = pickle.load(f)

    knn = meal_data['knn_model']
    scaler = meal_data['scaler']
    db_meal = meal_data['meal_db']
    features_model = meal_data['features'] # CRITICAL: These are the columns the model expects

    print("Model Meal Loaded Successfully.")
    print(f"   Features Expected: {features_model}")

except FileNotFoundError:
    print(f"Error: Model file not found at {file_path}")
    print("   Please verify the path or run the training script first.")
    exit()

Model Meal Loaded Successfully.
   Features Expected: ['Energy', 'Protein', 'Fat', 'Carbs']


## Meal Reccomendation

In [5]:
# ==============================================================================
# 2. MEAL PLAN GENERATION ENGINE
# ==============================================================================

def generate_full_day_plan(daily_target, meal_frequency):
    """
    Orchestrates the generation of a personalized full-day meal plan.

    This function breaks down total daily nutritional targets into specific 
    per-meal goals, queries the KNN model for matching food items, and 
    dynamically calibrates portion sizes to ensure calorie precision.

    ---------------------------------------------------------------------------
    Args:
        daily_target (dict): A dictionary containing the user's daily nutritional goals.
                              Required keys:
                              - 'Daily_Calories' (float): Total energy target (kcal).
                              - 'Target_Protein_g' (float): Total protein target (g).
                              - 'Target_Carbs_g' (float): Total carbohydrate target (g).
                              - 'Target_Fat_g' (float): Total fat target (g).

        meal_frequency (int): The number of meals to distribute the nutrition across 
                              (e.g., 3 for Breakfast, Lunch, Dinner).

    Returns:
        None: This function outputs the generated plan and summary directly 
              to the console standard output.

    ---------------------------------------------------------------------------
    Operational Logic:
    1.  Target Segmentation: 
        Divides the daily macro goals by the frequency (e.g., 2000 kcal / 4 meals = 500 kcal/meal).
    
    2.  Vector Alignment & Scaling: 
        Maps the inputs to the exact feature columns used during training ('Energy', 'Protein', etc.)
        and normalizes the values using the loaded Scaler.
    
    3.  Nearest Neighbor Search (KNN): 
        Queries the database for foods with the most similar nutritional profile to the per-meal target.
    
    4.  Diversity Filtering: 
        Maintains a 'blacklist' of items already selected for previous meals to ensure variety.
    
    5.  Smart Portioning: 
        Calculates a portion multiplier (e.g., 1.5x serving) to align the chosen food's 
        calories with the user's exact calorie target.
    """

    print("\n" + "="*50)
    print("GENERATING MEAL PLAN")
    print("="*50)
    
    # --- STEP 1: CALCULATE TARGET PER MEAL (AVERAGE) ---
    # Objective: Split the daily goal evenly across the number of meals.
    avg_cal = daily_target['Daily_Calories'] / meal_frequency
    avg_prot = daily_target['Target_Protein_g'] / meal_frequency
    avg_carbs = daily_target['Target_Carbs_g'] / meal_frequency
    avg_fat = daily_target['Target_Fat_g'] / meal_frequency

    print(f"DAILY TARGET : {daily_target['Daily_Calories']} kcal")
    print(f"FREQUENCY    : {meal_frequency}x meals")
    print(f"PER MEAL     : ~{avg_cal:.0f} kcal | {avg_prot:.1f}g Prot")
    print("-" * 50)
    
    # --- STEP 2: PREPARE INPUT VECTOR FOR AI ---
    # Objective: Format the data exactly how the model expects it during training.
    
    # Mapping user inputs to the model's specific feature column names.
    # This prevents shape mismatches during inference.
    input_data_map = {
        'Energy': avg_cal,
        'Protein': avg_prot,
        'Carbs': avg_carbs,
        'Fat': avg_fat,
    }
    
    # Convert to DataFrame and enforce column order
    try:
        input_df = pd.DataFrame([input_data_map])[features_model]
    except KeyError as e:
        print(f"ERROR: Column Name Mismatch. Your model expects {features_model}")
        print(f"   But we calculated these keys: {list(input_data_map.keys())}")
        return

    # Apply Scaling (Normalization)
    # The KNN calculates distance, so unscaled large numbers (Calories) would 
    # dominate small numbers (Fat) without this step.
    input_scaled = scaler.transform(input_df)
    
    # --- STEP 3: SEARCH LOOP (MEAL ITERATION) ---
    chosen_foods = [] # List to track eaten foods to prevent duplicates
    total_daily_cal = 0
    total_daily_prot = 0
    
    for i in range(1, meal_frequency + 1):
        
        # A. Neighbors Retrieval
        # We fetch top 10 neighbors to have a pool of candidates for diversity filtering.
        distances, indices = knn.kneighbors(input_scaled, n_neighbors=10)
        
        # Get candidate rows from the database
        candidates = db_meal.iloc[indices[0]].copy()
        
        # B. Diversity Filter
        # Exclude foods that appear in 'chosen_foods'
        candidates = candidates[~candidates['Food Items'].isin(chosen_foods)]
        
        # Fallback: If filtering removes all candidates (rare), reload the original list
        if candidates.empty:
            candidates = db_meal.iloc[indices[0]].copy()
        
        # C. Selection
        # Pick the best remaining match (Top 1)
        selected_item = candidates.iloc[0]
        
        # D. Smart Portioning Algorithm
        # Logic: If target is 500kcal but food is 250kcal, set portion to 2.0x
        portion = avg_cal / selected_item['Energy'] 
        
        # Round to nearest 0.5 (e.g., 1.0, 1.5, 2.0) for realistic serving sizes
        portion = round(portion * 2) / 2
        
        # Safety Caps: Ensure portions are neither too small nor absurdly large
        if portion < 0.5: portion = 0.5
        if portion > 3.0: portion = 3.0 
        
        # E. Record Selection
        chosen_foods.append(selected_item['Food Items'])
        
        # Calculate actual intake based on the adjusted portion
        intake_cal = selected_item['Energy'] * portion
        intake_prot = selected_item['Protein'] * portion
        
        total_daily_cal += intake_cal
        total_daily_prot += intake_prot
        
        # Output Meal Detail
        print(f"MEAL #{i}")
        print(f"   Menu    : {selected_item['Food Items']}")
        print(f"   Portion : {portion} x Serving ({selected_item['Energy']:.0f} kcal/srv)")
        print(f"   Total   : {intake_cal:.0f} kcal | {intake_prot:.1f}g Prot")
        print("-" * 30)

    # --- STEP 4: FINAL SUMMARY ---
    # Objective: Display the accumulated totals vs the original goals.
    print("="*50)
    print("DAILY SUMMARY")
    print(f"   Calories : {total_daily_cal:.0f} / {daily_target['Daily_Calories']} ({total_daily_cal/daily_target['Daily_Calories']:.0%})")
    print(f"   Protein  : {total_daily_prot:.0f}g / {daily_target['Target_Protein_g']}g")
    print("="*50)

## Execution

In [6]:
# ==============================================================================
# 3. TEST EXECUTION
# ==============================================================================

# Simulated user target input
user_target = {
    'Daily_Calories': 2200,
    'Target_Protein_g': 180,
    'Target_Carbs_g': 200,
    'Target_Fat_g': 70
}

# Run the function
generate_full_day_plan(user_target, meal_frequency=3)


GENERATING MEAL PLAN
DAILY TARGET : 2200 kcal
FREQUENCY    : 3x meals
PER MEAL     : ~733 kcal | 60.0g Prot
--------------------------------------------------
MEAL #1
   Menu    : Gun powder chutney
   Portion : 2.5 x Serving (312 kcal/srv)
   Total   : 781 kcal | 53.9g Prot
------------------------------
MEAL #2
   Menu    : Lobster Roll Sandwich
   Portion : 1.5 x Serving (450 kcal/srv)
   Total   : 675 kcal | 30.0g Prot
------------------------------
MEAL #3
   Menu    : Maa chaane ki dal
   Portion : 2.0 x Serving (345 kcal/srv)
   Total   : 689 kcal | 39.6g Prot
------------------------------
DAILY SUMMARY
   Calories : 2145 / 2200 (98%)
   Protein  : 123g / 180g
